# LSTM en GRU



In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Gebruik de GPU als die er is, anders de CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## voorspel instellingen

In [6]:
stap_per_dag = 24 * 12               # 288 per dag

aantal_dagen_input = 2               # aantal dagen voor input
input_len = aantal_dagen_input * stap_per_dag

aantal_dagen_forecast = 3            # =aantal dagen voorspelling
horizon = aantal_dagen_forecast * stap_per_dag

stride = 12                          # het venster schuift 12 stappen (1 uur) per voorbeeld op

TARGET_COL = ""                   # target
FEATURE_COLS = []                    # features (inclusief target)

TRAIN_FRAC = 0.70                    # deel van de data voor training
VAL_FRAC = 0.15                      # deel voor validatie data  ,de rest is test data

## Data laden
Eisen: opgeschoond, de index is de tijd (elke 5 minuten), geen dubbele tijdstippen.

In [4]:
df = None

In [5]:
n_features = len(FEATURE_COLS)
target_idx = FEATURE_COLS.index(TARGET_COL)

ValueError: 'POW' is not in list

## chronologisch splisen

In [ ]:
values = np.asarray(df[FEATURE_COLS])

n = len(values)
i_train = int(n * TRAIN_FRAC)
i_val = int(n * (TRAIN_FRAC + VAL_FRAC))

train_raw = values[:i_train]
val_raw = values[i_train:i_val]
test_raw = values[i_val:]

## normaliseren

In [ ]:
train_mean = train_raw.mean(axis=0)
train_std = train_raw.std(axis=0)

train = (train_raw - train_mean) / train_std
val = (val_raw - train_mean) / train_std
test = (test_raw - train_mean) / train_std

In [ ]:
class WindowDataset(Dataset):

    def __init__(self, data, input_len, horizon, target_idx, stride):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.input_len = input_len
        self.horizon = horizon
        self.target_idx = target_idx

        # Hoeveel vensters passen er in de data? Elk venster heeft input_len + horizon stappen nodig
        n_vensters = len(data) - input_len - horizon + 1
        self.starts = range(0, max(n_vensters, 0), stride)

    # Bereken het aantal voorbeelden
    def __len__(self):
        return len(self.starts)

    # Haal voorbeeld nummer idx op
    def __getitem__(self, idx):
        s = self.starts[idx]
        x = self.data[s : s + self.input_len]                                   # geschiedenis
        y = self.data[s + self.input_len : s + self.input_len + self.horizon,   # toekomst,
                      self.target_idx]                                          # alleen de target
        return x, y

Creeer dataset voor de training, validatie en test data.

In [ ]:
train_dataset = WindowDataset(train, input_len, horizon, target_idx, stride)
val_dataset = WindowDataset(val, input_len, horizon, target_idx, stride)
test_dataset = WindowDataset(test, input_len, horizon, target_idx, stride)

print("aantal voorbeelden | train:", len(train_dataset), "| val:", len(val_dataset), "| test:", len(test_dataset))

Creeer DataLoader 

In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

## model

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
        super(RNNModel, self).__init__()

        self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)

        # Van de uitvoer van de laatste stap naar de voorspelling
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x heeft de vorm (batch, tijd, features)
        out, _ = self.rnn(x)         
        out = out[:, -1, :]          
        return self.fc(out)          

## training loop


In [1]:
rnn_type = "lstm"               # "lstm" 
hidden_size = []             # het aantal getallen van hidden size staat gelijk aan het aantal lagen 
dropout = 0.2                  
learning_rate = 1e-3
output_size = horizon           

class CustomLSTM(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, dropout=0.0):
        super().__init__()

        self.layers = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        prev_size = input_size
        for h in hidden_sizes:
            self.layers.append(
                nn.LSTM(input_size=prev_size,
                        hidden_size=h,
                        num_layers=1,
                        batch_first=True)
            )
            prev_size = h

        self.fc = nn.Linear(hidden_sizes[-1], output_size)

    def forward(self, x):
        out = x
        for lstm in self.layers:
            out, _ = lstm(out)
            out = self.dropout(out)

        out = out[:, -1, :]
        return self.fc(out)
hidden_sizes = [12, 24, 48]   # of wat jij wilt

model = CustomLSTM(
    input_size=n_features,
    hidden_sizes=hidden_sizes,
    output_size=horizon,
    dropout=0.2
).to(DEVICE)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("CUSTOM LSTM |", sum(p.numel() for p in model.parameters()), "parameters")


NameError: name 'horizon' is not defined

In [ ]:
n_epochs = 30

# Houd de training performance bij
train_loss_list = []
val_loss_list = []

# Run the training loop
for epoch in range(n_epochs):

    # Trainen
    model.train()
    running_loss = 0.0

    # Iterate over the DataLoader for training data
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()                   # gradienten naar 0
        output = model(inputs)                  # voorspellen
        loss = criterion(output, targets)       # fout berekenen
        loss.backward()                         # backpropagation
        optimizer.step()                        # gewichten aanpassen

        # Houd de performance bij
        running_loss += loss.item() * targets.shape[0]

    # Bereken de loss over alle data samples
    train_loss = running_loss / len(train_dataset)
    train_loss_list.append(train_loss)

    # Evaluate the model performance on the validation dataset
    model.eval()
    val_loss = 0.0
    with torch.no_grad():                       # tijdens validatie niet leren
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            loss = criterion(model(inputs), targets)
            val_loss += loss.item() * targets.shape[0]
    val_loss = val_loss / len(val_dataset)
    val_loss_list.append(val_loss)

    print(f"Epoch {epoch+1}/{n_epochs}, train loss: {train_loss:.4f}, validation loss: {val_loss:.4f}")